# Linear and Regularized Regression

**Goal**: Predict a continuous target (a number) using features.

We will use a dataset, split it into training data (to teach our model) and testing data (to check how well it learned), and compare basic Linear Regression with two regularized versions: Lasso and Ridge.

### 1. Load the Dataset

We will use the Diabetes dataset from `scikit-learn`. It contains physiological features (like age, bmi, blood pressure) to predict disease progression.

In [1]:
import pandas as pd
from sklearn.datasets import load_diabetes

data = load_diabetes()
print("Features:", data.feature_names)
print("Data shape:", data.data.shape)

Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Data shape: (442, 10)


### 2. Brief Exploration

Let's put the data into a pandas DataFrame to see what it looks like.

In [2]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [3]:
df.describe()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,442.000000
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.268604e-17,1.130318e-17,152.133484
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,77.093005
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01,25.000000
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02,87.000000
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03,140.500000
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02,211.500000
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01,346.000000


### 3. Split into Features (X) and Target (y)

We separate the columns we use to predict (`X`) from the column we want to predict (`y`).

In [4]:
X = df.drop(columns=['target'])
y = df['target']

### 4. Train/Test Split

We split our data so we can train the model on one portion (e.g., 80%) and evaluate its true performance on the unseen portion (20%).

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (353, 10)
Test size: (89, 10)


### 5. Baseline Model: Linear Regression

We'll fit a basic linear regression model and predict.

In [6]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


Now we make predictions on both the training set and the test set.

In [7]:
y_pred_train_lr = model_lr.predict(X_train)
y_pred_test_lr = model_lr.predict(X_test)

### Evaluate Linear Regression

We use Mean Squared Error (MSE) and R-squared to evaluate. Lower MSE is better. R-squared closer to 1 is better.

In [8]:
from sklearn.metrics import mean_squared_error, r2_score

print("LR Train MSE:", mean_squared_error(y_train, y_pred_train_lr))
print("LR Test MSE:", mean_squared_error(y_test, y_pred_test_lr))
print("LR Test R2:", r2_score(y_test, y_pred_test_lr))

LR Train MSE: 2868.549702835578
LR Test MSE: 2900.1936284934814
LR Test R2: 0.4526027629719195


### 6. Lasso Regression (L1 Regularization)

Lasso adds a penalty to the model for having large coefficients. It can even shrink some coefficients exactly to 0, completely ignoring some features!

In [9]:
from sklearn.linear_model import Lasso

model_lasso = Lasso(alpha=0.1)
model_lasso.fit(X_train, y_train)

y_pred_train_las = model_lasso.predict(X_train)
y_pred_test_las = model_lasso.predict(X_test)

Now let's check Lasso's performance.

In [10]:
print("Lasso Train MSE:", mean_squared_error(y_train, y_pred_train_las))
print("Lasso Test MSE:", mean_squared_error(y_test, y_pred_test_las))
print("Lasso Test R2:", r2_score(y_test, y_pred_test_las))

Lasso Train MSE: 2935.25823259759
Lasso Test MSE: 2798.1934851697188
Lasso Test R2: 0.4718547867276227


### 7. Ridge Regression (L2 Regularization)

Ridge also adds a penalty for large coefficients, but it rarely forces them exactly to zero. It just makes them smaller to prevent overfitting.

In [11]:
from sklearn.linear_model import Ridge

model_ridge = Ridge(alpha=0.1)
model_ridge.fit(X_train, y_train)

y_pred_train_rdg = model_ridge.predict(X_train)
y_pred_test_rdg = model_ridge.predict(X_test)

Now let's check Ridge's performance.

In [12]:
print("Ridge Train MSE:", mean_squared_error(y_train, y_pred_train_rdg))
print("Ridge Test MSE:", mean_squared_error(y_test, y_pred_test_rdg))
print("Ridge Test R2:", r2_score(y_test, y_pred_test_rdg))

Ridge Train MSE: 2912.9835415879015
Ridge Test MSE: 2856.4868876706537
Ridge Test R2: 0.46085219464119265


### 8. Simple Comparison

Let's compare the testing R2 scores side by side. Regularization helps control overfitting, improving generalization to the test set.

In [13]:
print("Linear Regression R2:", r2_score(y_test, y_pred_test_lr))
print("Lasso Regression R2:", r2_score(y_test, y_pred_test_las))
print("Ridge Regression R2:", r2_score(y_test, y_pred_test_rdg))

Linear Regression R2: 0.4526027629719195
Lasso Regression R2: 0.4718547867276227
Ridge Regression R2: 0.46085219464119265


### 9. Comparing Coefficients (Feature Selection)

A key difference between L1 (Lasso) and L2 (Ridge) regularization is how they handle the model's learned coefficients (the weights given to each feature):
- **Lasso (L1)** can push coefficients to **exactly zero**. This acts as automatic **feature selection**, dropping less important features entirely!
- **Ridge (L2)** pushes coefficients *close* to zero to prevent overfitting, but rarely makes them exactly zero. It keeps all features but softly reduces their impact.

Let's look at the coefficients for all three models side-by-side to see this in action. Notice how Lasso zeros out some coefficients, while Ridge just shrinks them compared to plain Linear Regression.

In [ ]:
# Create a DataFrame to compare coefficients
coef_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Linear Regression': model_lr.coef_,
    'Lasso (L1)': model_lasso.coef_,
    'Ridge (L2)': model_ridge.coef_
})

# Show the dataframe rounded to 2 decimal places for readability
coef_df.round(2)

### Try It Yourself!

**Exercises:**
1. Go to the Train/Test Split cell, change `test_size=0.2` to `test_size=0.4`, and re-run all cells. How do the results change?
2. Go to the Lasso and Ridge cells, change `alpha=0.1` to a larger value like `1.0` or `10.0`. What happens to the R2 score? Does the penalty get too strict?